# Station Stacking v16 - KATL

Experimental notebook for `KATL`.

V16 is a single fused-weather experiment: strict curated v11-style base plus the eight forecast-temp and precip/cloud aggregate weather features tested in v15.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TIMING_MODE = "same_day_11am_live_safe"
TARGET_SOURCE = "iem_hourly"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
EXPORT_MODEL_WEIGHTS = True
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v16"
V15_OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v15"
MODEL_VERSION = "station_high_regressor_v16_fused_weather_stack"

PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from scripts.run_station_stacking_v16 import write_v15_comparisons
from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V16_ADDITIONAL_FEATURE_COLUMNS,
    V16_BLOCKED_BASE_FEATURE_COLUMNS,
    V16_DROPPED_FEATURE_COLUMNS,
    V16_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V16 Contract

`feature_version="v16_fused"` uses the strict curated v11-style base, excludes accidental raw/provider weather sprawl, and adds only the eight fused v13 aggregate weather features when train-year coverage passes.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

{
    "folds": fold_spec,
    "v16_feature_count": len(V16_FEATURE_COLUMNS),
    "v16_additional_features": V16_ADDITIONAL_FEATURE_COLUMNS,
    "v16_blocked_base_features": sorted(V16_BLOCKED_BASE_FEATURE_COLUMNS),
    "v16_dropped_features": sorted(V16_DROPPED_FEATURE_COLUMNS),
}


{'folds':                      fold  train_start_year  train_end_year  validation_year
 0  fold_2021_2023_to_2024              2021            2023             2024
 1  fold_2021_2024_to_2025              2021            2024             2025,
 'v16_feature_count': 64,
 'v16_additional_features': ['v13_forecast_temp_at_as_of_mean_f',
  'v13_forecast_temp_at_as_of_minus_observed_mean_f',
  'v13_forecast_temp_at_as_of_spread_f',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'v13_cloud_cover_mean_pct',
  'v13_cloud_cover_max_pct',
  'v13_cloud_cover_remaining_warmup_interaction',
  'v13_precip_cloud_remaining_warmup_interaction'],
 'v16_blocked_base_features': ['v8_cloud_cover_max_remaining_warmup_interaction',
  'v8_cloud_cover_mean_remaining_warmup_interaction'],
 'v16_dropped_features': ['actual_minus_climatology_10y_f_DIAGNOSTIC_ONLY',
  'climatology_source_end_year',
  'climatology_source_start_year',
  'gfs_as_of_hour_local',
  'gfs_available',
  'gfs_forecast_window_ho

## Data Availability


In [4]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1998,2021-01-01,2026-06-21
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1997,2021-01-01,2026-06-21


## Run Fused Model


In [5]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v16_fused",
    target_mode="remaining_warmup",
    target_source=TARGET_SOURCE,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=OUTPUT_DIR / "fused",
    climatology_normals_path=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9" / "station_rolling_10y_daily_high_normals.csv",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v16/fused/KATL_optuna.sqlite3')

In [6]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:2673: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2672: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2673: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,728,1.648918,2.369137
1,validation_2024_2025,lightgbm,728,1.679695,2.413078
2,validation_2024_2025,catboost,728,1.664575,2.380856
3,validation_2024_2025,provider_mean,728,2.832206,4.006255
4,validation_2024_2025,provider_median,728,2.782240,3.911262
5,validation_2024_2025,nbm_raw,728,2.792647,3.816647
6,validation_2024_2025,hrrr_raw,728,3.556181,4.953880
7,validation_2024_2025,gfs_raw,728,3.254687,4.459385
8,test_2026,xgboost,170,1.525665,2.143606
9,test_2026,lightgbm,170,1.584410,2.219679


In [7]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v16",
    )
    display((exported_weights.bundle_path, exported_weights.manifest_path))

write_v15_comparisons(OUTPUT_DIR, V15_OUTPUT_DIR, STATION_ID)


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v16/fused/model_weights/KATL_station_high_regressor_v16_fused_weather_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v16/fused/model_weights/KATL_station_high_regressor_v16_fused_weather_stack.json'))

Wrote KATL v16/v15 comparison: D:\dev\weather-research\data\calibration\station_stacking_v16\KATL_v16_fused_vs_v15_common_date_comparison.csv


## V15 Reference Comparison


In [8]:
comparison_path = OUTPUT_DIR / f"{STATION_ID}_v16_fused_vs_v15_common_date_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
comparison.sort_values(["method", "delta_mae_f", "reference"]) if not comparison.empty else comparison


,station_id,reference,comparison,feature_version,model_version,method,common_date_count,reference_mae_f,fused_mae_f,delta_mae_f,reference_rmse_f,fused_rmse_f,delta_rmse_f,fused_better_days,reference_better_days,tied_days,actual_mismatch_count,first_common_date,last_common_date
18,KATL,v15_precip_cloud,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,catboost,170,1.623187,1.664224,0.041037,2.292603,2.299231,0.006628,77,93,0,0,2026-01-01,2026-06-21
0,KATL,v15_base,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,catboost,170,1.582195,1.664224,0.082029,2.211646,2.299231,0.087585,76,94,0,0,2026-01-01,2026-06-21
9,KATL,v15_forecast_temp_at_as_of,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,catboost,170,1.487267,1.664224,0.176957,2.134523,2.299231,0.164708,76,94,0,0,2026-01-01,2026-06-21
1,KATL,v15_base,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
10,KATL,v15_forecast_temp_at_as_of,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
19,KATL,v15_precip_cloud,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
2,KATL,v15_base,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
11,KATL,v15_forecast_temp_at_as_of,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
20,KATL,v15_precip_cloud,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
12,KATL,v15_forecast_temp_at_as_of,v16_fused,v16_fused,station_high_regressor_v16_fused_weather_stack,lightgbm,170,1.569194,1.584410,0.015216,2.208089,2.219679,0.011590,79,91,0,0,2026-01-01,2026-06-21


## Selected Feature Audit


In [9]:
selected = set(result.feature_columns["feature"].astype(str))
{
    "selected_feature_count": len(selected),
    "v16_additional_selected": sorted(selected & set(V16_ADDITIONAL_FEATURE_COLUMNS)),
    "blocked_base_selected": sorted(selected & set(V16_BLOCKED_BASE_FEATURE_COLUMNS)),
    "unexpected_v13_selected": sorted(
        feature
        for feature in selected
        if feature.startswith("v13_") and feature not in set(V16_ADDITIONAL_FEATURE_COLUMNS)
    ),
}


{'selected_feature_count': 223,
 'v16_additional_selected': ['v13_cloud_cover_max_pct',
  'v13_cloud_cover_mean_pct',
  'v13_cloud_cover_remaining_warmup_interaction',
  'v13_forecast_temp_at_as_of_mean_f',
  'v13_forecast_temp_at_as_of_minus_observed_mean_f',
  'v13_forecast_temp_at_as_of_spread_f',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'v13_precip_cloud_remaining_warmup_interaction'],
 'blocked_base_selected': [],
 'unexpected_v13_selected': []}

## Added Feature Coverage


In [10]:
available = [column for column in V16_ADDITIONAL_FEATURE_COLUMNS if column in result.features]
added_feature_coverage = (
    result.features[available]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
) if available else pd.DataFrame()

added_feature_coverage.sort_values("coverage_pct", ascending=False)


,feature,coverage_pct
0,v13_forecast_temp_at_as_of_mean_f,100.000000
1,v13_forecast_temp_at_as_of_minus_observed_mean_f,100.000000
2,v13_forecast_temp_at_as_of_spread_f,100.000000
3,v13_forecast_temp_bias_remaining_warmup_intera...,100.000000
4,v13_cloud_cover_mean_pct,66.016016
5,v13_cloud_cover_max_pct,66.016016
6,v13_cloud_cover_remaining_warmup_interaction,66.016016
7,v13_precip_cloud_remaining_warmup_interaction,65.915916


## Raw Weather Sprawl Check


In [11]:
raw_weather_tokens = (
    "cloud",
    "ceiling",
    "dewpoint",
    "forecast_temp_at_as_of",
    "humidity",
    "precip",
    "pressure",
    "shortwave",
    "visibility",
    "wind_",
)
selected_features = result.feature_columns["feature"].astype(str)
raw_weather_selected = pd.DataFrame(
    [
        {"feature": feature}
        for feature in selected_features
        if feature.startswith(("gfs_", "hrrr_", "nbm_"))
        and any(token in feature for token in raw_weather_tokens)
    ]
)

raw_weather_selected


""


## Rounded Within 1F Accuracy


In [12]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)
predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
8,oof_2026,xgboost,170,104,61.176471
7,oof_2026,ridge_stack,170,103,60.588235
3,oof_2026,lightgbm,170,101,59.411765
0,oof_2026,catboost,170,95,55.882353
4,oof_2026,nbm_raw,170,68,40.000000
5,oof_2026,provider_mean,170,63,37.058824
6,oof_2026,provider_median,170,63,37.058824
2,oof_2026,hrrr_raw,170,56,32.941176
1,oof_2026,gfs_raw,170,47,27.647059
9,validation_2024_2025,catboost,728,436,59.890110


## Bracket Metrics


In [13]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,170,1.525665,2.143606,44.117647
1,lightgbm,170,1.584410,2.219679,41.764706
2,catboost,170,1.664224,2.299231,39.411765
3,ridge_stack,170,1.537176,2.150350,42.941176
4,provider_mean,170,2.786086,4.048184,25.294118
5,provider_median,170,2.723802,3.926838,27.647059
6,nbm_raw,170,2.614555,3.750025,28.823529
7,hrrr_raw,170,3.216669,4.676677,24.705882
8,gfs_raw,170,3.328938,4.532080,20.588235
